In [1]:
!python -c 'import torch;print(torch.__version__)'
# Check nvcc version
!nvcc -V
# Check GCC version
!gcc --version
!nvidia-smi

2.9.0+cu128
/bin/bash: 줄 1: nvcc: 명령어를 찾을 수 없음
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Mon Mar  9 06:11:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 

# MMDET

In [3]:
import sys
# Point this exactly to your mmdetection root folder
sys.path.append('/mnt/Documents/Dad/github/DUP/mmdetection_new')

from mmdet.apis import init_detector

# 1. Path to the config file you used for training
# config_file = '/mnt/Documents/Dad/github/DUP/mmdetection_new/configs/_config_2026/UDP_faster_rcnn_resnet50_TrainVal_2.py'
config_file = '/mnt/Documents/Dad/github/DUP/mmdetection_new/configs/_config_2026/UDP_centernet_resnet50_TrainVal.py'
# 2. Build the model
model = init_detector(config_file, device='cpu')

# 3. Calculate total parameters
total_params = sum(p.numel() for p in model.parameters())

print(f"Total Parameters: {total_params:,}")

Total Parameters: 32,115,532


/mnt/Documents/Dad/github/DUP/mmdetection_new/mmdet/apis/inference.py:70: UserWarning: checkpoint is None, use COCO classes by default.
  warnings.warn('checkpoint is None, use COCO classes by default.')


In [16]:
!python /mnt/Documents/Dad/github/DUP/mmdetection_new/fps_test_centernet.py

Loads checkpoint by local backend from path: /mnt/Documents/Dad/github/DUP/mmdetection_new/work_dirs/centernet/centernet_r50_TrainVal_epoch_8/seed0/epoch_8.pth
Warming up...
/home/hj/anaconda3/envs/mmdet_v3/lib/python3.9/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1695392035629/work/aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Starting timing loop for 100 iterations...

--- Results ---
Total time for 100 runs: 0.7625s
Average inference time: 0.0076s
FPS: 131.15


In [14]:
!python /mnt/Documents/Dad/github/DUP/mmdetection_new/fps_test_fasterrcnn.py

Loads checkpoint by local backend from path: /mnt/Documents/Dad/github/DUP/mmdetection_new/work_dirs/fasterrcnn/faster_rcnn_r50_TrainVal_epoch_13/seed0/epoch_13.pth
Warming up...
Starting timing loop for 100 iterations...

--- Results ---
Total time for 100 runs: 0.8888s
Average inference time: 0.0089s
FPS: 112.51


# YOLO

In [3]:
%cd /mnt/Documents/Dad/github/DUP/yolov9_udp

/mnt/Documents/Dad/github/DUP/yolov9_udp


In [4]:
import time
import torch
import cv2
import numpy as np
from models.common import DetectMultiBackend
from utils.general import non_max_suppression, check_img_size
from utils.augmentations import letterbox

# ================= CONFIGURATION =================
# Update this to your YOLOv9 weights path
WEIGHTS = '/mnt/Documents/Dad/github/DUP/yolov9_udp/results_T_ITS_2026/Euljiro/1_2_2_Euljiro_off_peak_balanced_TrainVal_gelan-c_seed1/weights/last.pt'  
# Path to the same test image
IMAGE_PATH = '/mnt/Documents/Dad/github/DUP/DATA/Euljiro/1_balanced_simulation/test/4_025950.jpg'
DEVICE = 'cuda:0'
IMG_SIZE = (320, 320) # Standard YOLO size
# =================================================

def run_benchmark():
    device = torch.device(DEVICE)
    print(f"Loading YOLOv9 model from {WEIGHTS}...")
    
    # 1. Load Model
    model = DetectMultiBackend(WEIGHTS, device=device, dnn=False, data=None, fp16=False)
    stride, names, pt = model.stride, model.names, model.pt
    imgsz = check_img_size(IMG_SIZE, s=stride)  # check image size

    # 2. Prepare Image (Pre-process once to measure pure inference)
    # Read image
    img0 = cv2.imread(IMAGE_PATH)
    # Padded resize
    img = letterbox(img0, imgsz, stride=stride, auto=pt)[0]
    # Convert
    img = img.transpose((2, 0, 1))[::-1]  # HWC to CHW, BGR to RGB
    img = np.ascontiguousarray(img)
    img = torch.from_numpy(img).to(device)
    img = img.float()  # uint8 to fp16/32
    img /= 255.0  # 0 - 255 to 0.0 - 1.0
    if len(img.shape) == 3:
        img = img[None]  # expand for batch dim

    # 3. Warm-up
    print("Warming up (10 runs)...")
    model.warmup(imgsz=(1, 3, *imgsz))  # warmup
    for _ in range(10):
        _ = model(img)

    # 4. Benchmark Loop
    print("Benchmarking (100 runs)...")
    t_start = time.time()
    for _ in range(100):
        pred = model(img)
        # We strictly measure inference time. NMS is usually post-processing.
        # If you want to include NMS in the speed, uncomment the next line:
        # pred = non_max_suppression(pred, 0.25, 0.45, classes=None, agnostic=False, max_det=1000)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.time()

    # 5. Results
    avg_time = (t_end - t_start) / 100
    fps = 1.0 / avg_time
    
    print(f"\n=== YOLOv9 RESULTS ===")
    print(f"Average Inference Time: {avg_time:.5f} s")
    print(f"FPS: {fps:.2f}")

if __name__ == "__main__":
    run_benchmark()

Loading YOLOv9 model from /mnt/Documents/Dad/github/DUP/yolov9_udp/results_T_ITS_2026/Euljiro/1_2_2_Euljiro_off_peak_balanced_TrainVal_gelan-c_seed1/weights/last.pt...


Fusing layers... 
udp_gelan-c summary: 387 layers, 25229401 parameters, 0 gradients


Warming up (10 runs)...
Benchmarking (100 runs)...

=== YOLOv9 RESULTS ===
Average Inference Time: 0.00358 s
FPS: 279.07


In [5]:
%cd /mnt/Documents/Dad/github/DUP/yolov7_dup

/mnt/Documents/Dad/github/DUP/yolov7_dup


In [8]:
import sys
import os
import time
import torch
import cv2
import numpy as np

# ================= CRITICAL FIX FOR NOTEBOOKS =================
YOLOV7_ROOT = '/mnt/Documents/Dad/github/DUP/yolov7_dup'

# 1. Force Python to "forget" conflicting modules
for key in list(sys.modules.keys()):
    if key.startswith('utils') or key.startswith('models'):
        del sys.modules[key]

# 2. Insert the YOLOv7 path at the VERY FRONT
if YOLOV7_ROOT not in sys.path:
    sys.path.insert(0, YOLOV7_ROOT)

print(f"Set Python path to: {YOLOV7_ROOT}")
# ==============================================================

# Now imports should work
from utils.general import check_img_size
from utils.datasets import letterbox 

# ================= CONFIGURATION =================
WEIGHTS = '/mnt/Documents/Dad/github/DUP/yolov7_dup/runs/train/2_yolov7_tiny_seed_0/weights/last.pt'
IMAGE_PATH = '/mnt/Documents/Dad/github/DUP/DATA/Euljiro/1_balanced_simulation/test/4_025950.jpg'
DEVICE = 'cuda:0'
IMG_SIZE = 320
# =================================================

def run_benchmark():
    device = torch.device(DEVICE)
    print(f"Loading YOLOv7 model from {WEIGHTS}...")

    try:
        # Load checkpoint with weights_only=False
        ckpt = torch.load(WEIGHTS, map_location=device, weights_only=False)
        
        model = None
        
        # --- ROBUST MODEL EXTRACTION ---
        if isinstance(ckpt, dict):
            # 1. Try EMA first, but check if it's actually valid
            if 'ema' in ckpt and ckpt['ema'] is not None:
                model = ckpt['ema']
                print("Loaded EMA model.")
            # 2. If EMA is missing/None, try standard model
            elif 'model' in ckpt:
                model = ckpt['model']
                print("Loaded standard model (EMA was None or missing).")
        
        # 3. If model is still None, maybe the checkpoint IS the model
        if model is None:
            model = ckpt
            print("Loaded raw model object.")
        
        # Final safety check
        if model is None:
            raise ValueError("Failed to extract any valid model from checkpoint.")

        # Set to evaluation mode
        model = model.float().to(device)
        if hasattr(model, 'fuse'):
            model.fuse()
        model.eval()
            
    except Exception as e:
        print(f"Error loading model directly: {e}")
        import traceback
        traceback.print_exc()
        return

    # Check stride
    stride = int(model.stride.max()) if hasattr(model, 'stride') else 32
    imgsz = check_img_size(IMG_SIZE, s=stride)

    # 2. Prepare Image
    img0 = cv2.imread(IMAGE_PATH)
    if img0 is None:
        print(f"Error: Could not read image at {IMAGE_PATH}")
        return

    img = letterbox(img0, imgsz, stride=stride)[0]
    img = img[:, :, ::-1].transpose(2, 0, 1)  # BGR to RGB
    img = np.ascontiguousarray(img)
    img = torch.from_numpy(img).to(device)
    img = img.float()
    img /= 255.0
    if img.ndimension() == 3:
        img = img.unsqueeze(0)

    # 3. Warm-up
    print("Warming up (10 runs)...")
    with torch.no_grad():
        for _ in range(10):
            _ = model(img)

    # 4. Benchmark Loop
    print("Benchmarking (100 runs)...")
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    t_start = time.time()
    with torch.no_grad():
        for _ in range(100):
            _ = model(img)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.time()

    # 5. Results
    avg_time = (t_end - t_start) / 100
    fps = 1.0 / avg_time
    
    print(f"\n=== YOLOv7-tiny RESULTS ===")
    print(f"Average Inference Time: {avg_time:.5f} s")
    print(f"FPS: {fps:.2f}")

if __name__ == "__main__":
    run_benchmark()

Set Python path to: /mnt/Documents/Dad/github/DUP/yolov7_dup
Loading YOLOv7 model from /mnt/Documents/Dad/github/DUP/yolov7_dup/runs/train/2_yolov7_tiny_seed_0/weights/last.pt...
Loaded standard model (EMA was None or missing).
Fusing layers... 
IDetect.fuse
Warming up (10 runs)...
Benchmarking (100 runs)...

=== YOLOv7-tiny RESULTS ===
Average Inference Time: 0.00158 s
FPS: 632.32


In [1]:
!python /mnt/Documents/Dad/github/DUP/yolov7_dup/detect.py --weights /mnt/Documents/Dad/github/DUP/yolov7_dup/runs/train/2_yolov7_tiny_seed_4/weights/last.pt

Namespace(weights=['/mnt/Documents/Dad/github/DUP/yolov7_dup/runs/train/2_yolov7_tiny_seed_4/weights/last.pt'], source='inference/images', img_size=640, conf_thres=0.25, iou_thres=0.45, device='', view_img=False, save_txt=False, save_conf=False, nosave=False, classes=None, agnostic_nms=False, augment=False, update=False, project='runs/detect', name='exp', exist_ok=False, no_trace=False)
YOLOR 🚀 d01e7177 torch 2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080, 15926.6875MB)

Fusing layers... 
IDetect.fuse
Model Summary: 208 layers, 6013008 parameters, 0 gradients
 Convert model to Traced-model... 
 traced_script_module saved! 
 model is traced! 

Traceback (most recent call last):
  File "/mnt/Documents/Dad/github/DUP/yolov7_dup/detect.py", line 196, in <module>
    detect()
    ~~~~~~^^
  File "/mnt/Documents/Dad/github/DUP/yolov7_dup/detect.py", line 57, in detect
    dataset = LoadImages(source, img_size=imgsz, stride=stride)
  File "/mnt/Documents/Dad/github/DUP/yolov7_dup/utils/datasets.

In [32]:
import sys
import os
import time
import torch
import cv2
import numpy as np

# ================= CRITICAL FIX FOR NOTEBOOKS =================
sd_net_ROOT = '/mnt/Documents/Dad/github/DUP/yolov7_dup'

# 1. Force Python to "forget" conflicting modules
for key in list(sys.modules.keys()):
    if key.startswith('utils') or key.startswith('models'):
        del sys.modules[key]

# 2. Insert the SD-Net path at the VERY FRONT
if sd_net_ROOT not in sys.path:
    sys.path.insert(0, sd_net_ROOT)

print(f"Set Python path to: {sd_net_ROOT}")
# ==============================================================

# Now imports should work
from utils.general import check_img_size
from utils.datasets import letterbox 

# ================= CONFIGURATION =================
WEIGHTS    = '/mnt/Documents/Dad/github/DUP/yolov7_dup/runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed0/weights/last.pt'
IMAGE_PATH = '/mnt/Documents/Dad/github/DUP/DATA/Euljiro/1_balanced_simulation/test/4_025950.jpg'
DEVICE     = 'cuda:0'
IMG_SIZE   = 320
# =================================================

def run_benchmark():
    device = torch.device(DEVICE)
    print(f"Loading SD-Net model from {WEIGHTS}...")

    try:
        # Load checkpoint with weights_only=False
        ckpt = torch.load(WEIGHTS, map_location=device, weights_only=False)
        
        model = None
        
        # --- ROBUST MODEL EXTRACTION ---
        if isinstance(ckpt, dict):
            # 1. Try EMA first, but check if it's actually valid
            if 'ema' in ckpt and ckpt['ema'] is not None:
                model = ckpt['ema']
                print("Loaded EMA model.")
            # 2. If EMA is missing/None, try standard model
            elif 'model' in ckpt:
                model = ckpt['model']
                print("Loaded standard model (EMA was None or missing).")
        
        # 3. If model is still None, maybe the checkpoint IS the model
        if model is None:
            model = ckpt
            print("Loaded raw model object.")
        
        # Final safety check
        if model is None:
            raise ValueError("Failed to extract any valid model from checkpoint.")

        # Set to evaluation mode
        model = model.float().to(device)
        if hasattr(model, 'fuse'):
            model.fuse()
        model.eval()
            
    except Exception as e:
        print(f"Error loading model directly: {e}")
        import traceback
        traceback.print_exc()
        return

    # Check stride
    stride = int(model.stride.max()) if hasattr(model, 'stride') else 32
    imgsz = check_img_size(IMG_SIZE, s=stride)

    # 2. Prepare Image
    img0 = cv2.imread(IMAGE_PATH)
    if img0 is None:
        print(f"Error: Could not read image at {IMAGE_PATH}")
        return

    img = letterbox(img0, imgsz, stride=stride)[0]
    img = img[:, :, ::-1].transpose(2, 0, 1)  # BGR to RGB
    img = np.ascontiguousarray(img)
    img = torch.from_numpy(img).to(device)
    img = img.float()
    img /= 255.0
    if img.ndimension() == 3:
        img = img.unsqueeze(0)

    # 3. Warm-up
    print("Warming up (10 runs)...")
    with torch.no_grad():
        for _ in range(10):
            _ = model(img)

    # 4. Benchmark Loop
    print("Benchmarking (100 runs)...")
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    t_start = time.time()
    with torch.no_grad():
        for _ in range(100):
            _ = model(img)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.time()

    # 5. Results
    avg_time = (t_end - t_start) / 100
    fps = 1.0 / avg_time
    
    print(f"\n=== sd_net RESULTS ===")
    print(f"Average Inference Time: {avg_time:.5f} s")
    print(f"FPS: {fps:.2f}")

if __name__ == "__main__":
    run_benchmark()

Set Python path to: /mnt/Documents/Dad/github/DUP/yolov7_dup
Loading SD-Net model from /mnt/Documents/Dad/github/DUP/yolov7_dup/runs/train/SD_net_TrainVal_2h6a_from_scratch_320_seed0/weights/last.pt...
Loaded standard model (EMA was None or missing).
Fusing layers... 
IDetect.fuse
Warming up (10 runs)...
Benchmarking (100 runs)...

=== sd_net RESULTS ===
Average Inference Time: 0.00099 s
FPS: 1012.74
